# 1. Overview

Reproduce the iAL1080 N–O2 growth/PHB phase planes and the FVA comparison of the predefined growth- and storage-dominant states. All numerical values are loaded from the four CSV files in `../data/`.

## 2. Load source data

In [ ]:
from pathlib import Path
import importlib.util
import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
package = None
for candidate in candidates:
    direct = candidate / 'scripts' / 'run_fva_phaseplane_figures.py'
    nested = candidate / 'analysis' / 'phb_n_o2_fva' / 'scripts' / 'run_fva_phaseplane_figures.py'
    if direct.is_file():
        package = candidate
        script = direct
        break
    if nested.is_file():
        package = candidate / 'analysis' / 'phb_n_o2_fva'
        script = nested
        break
if package is None:
    raise FileNotFoundError('Could not locate the phb_n_o2_fva package')

spec = importlib.util.spec_from_file_location('fva_reproduction', script)
fva_reproduction = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fva_reproduction)
fva_reproduction.configure_style()
tables = fva_reproduction.load_and_validate()
{k: v.shape for k, v in tables.items()}

## 3. Four-state sanity check

In [ ]:
states = tables['states'].set_index('State')
display(states)
combined = states.loc['N + low O2']
n_only = states.loc['N-limited']
assert np.isclose(combined['mu_max_h-1'], n_only['mu_max_h-1'])
assert combined['PHB_capacity_g_gDW_h'] <= n_only['PHB_capacity_g_gDW_h'] + 1e-6

## 4. N–O2 growth phase plane

In [ ]:
phase = tables['phase']
growth = phase.pivot(index='N_fraction_qNref', columns='O2_fraction_qO2ref', values='mu_max_h-1').sort_index().sort_index(axis=1)
x, y = np.meshgrid(growth.columns, growth.index)
fig, ax = plt.subplots(figsize=(5.2, 4.2))
image = ax.contourf(x, y, growth.to_numpy(), levels=18, cmap='Blues')
fig.colorbar(image, ax=ax, label='Maximum growth rate (h$^{-1}$)')
ax.set(xlabel=r'Relative oxygen uptake constraint ($q_{O_2}/q_{O_2,ref}$)', ylabel=r'Relative ammonium uptake constraint ($q_N/q_{N,ref}$)', title='Maximum growth capacity across the N–O$_2$ plane')
plt.show()

## 5. N–O2 PHB phase plane

In [ ]:
phb = phase.pivot(index='N_fraction_qNref', columns='O2_fraction_qO2ref', values='PHB_capacity_g_gDW_h').sort_index().sort_index(axis=1)
fig, ax = plt.subplots(figsize=(5.2, 4.2))
image = ax.contourf(x, y, phb.to_numpy(), levels=18, cmap='BuGn')
fig.colorbar(image, ax=ax, label='Maximum feasible PHB synthesis rate (g gDW$^{-1}$ h$^{-1}$)')
ax.set(xlabel=r'Relative oxygen uptake constraint ($q_{O_2}/q_{O_2,ref}$)', ylabel=r'Relative ammonium uptake constraint ($q_N/q_{N,ref}$)', title='Maximum PHB storage capacity across the N–O$_2$ plane')
plt.show()

## 6. FVA interval comparison

In [ ]:
fva = tables['fva']
positions = np.arange(len(fva))[::-1]
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for position, row in zip(positions, fva.itertuples(index=False)):
    ax.plot([row.reference_min_norm, row.reference_max_norm], [position + 0.12] * 2, lw=5, solid_capstyle='round', color='#7FA9C9')
    ax.plot([row.storage_min_norm, row.storage_max_norm], [position - 0.12] * 2, lw=5, solid_capstyle='round', color='#D79A84')
ax.set_yticks(positions, [fva_reproduction.REACTION_LABELS[x] for x in fva['reaction_id']])
ax.set_xlabel('Feasible flux interval (normalized within each reaction)')
ax.set_title('FVA-supported shift in feasible flux space between defined states')
plt.show()

## 7. Build the three standalone manuscript figures

In [ ]:
standalone_figures = fva_reproduction.make_all_figures(tables)
for figure_name, figure in standalone_figures.items():
    print(figure_name)
    display(figure)

## 8. Export

In [ ]:
output_dir = package / 'figures'
for figure_name, figure in standalone_figures.items():
    fva_reproduction.save_figure(figure, figure_name, output_dir)

## 9. Numerical summary

In [ ]:
fva_reproduction.print_summary(tables)
display(tables['crosscheck'])